# 02 -- Train SFT (Step 5) and put its checkpoints on the Elo-vs-compute curve (Step 6)

Fine-tune Qwen3-0.6B (LoRA adapters + value head) to imitate Stockfish, then rate a few saved checkpoints
against the frozen reference set into `results/results.csv` -- one row per checkpoint, with the training
GPU-hours it took to get there (plan v4, Sec 0).

Settings live in `configs/sft.yaml` (`policy:` = the player, `train:` = training) and `configs/eval.yaml`.

**Step 5 is done** (run `structured_masked_nall_e10_s0`, stopped on purpose at step 25,000 = 7.69 GPU-h), so
`RUN_TRAINING = False` by default: the notebook skips straight to the curves and the evaluation. Set it to
`True` only to train a new config (Step 8 runs one SFT per combo) -- with `SMOKE_TEST = True` first (64
positions, a few steps) to check speed and memory.

**Step 14 (plan v5): LLM SFT on the 500k dataset.** Same notebook, other config: run it headless with
`SFT_CONFIG=configs/sft_500k.yaml RUN_TRAINING=1 RUN_EVAL=0` (smoke test first with `SMOKE_TEST=1`;
`docs/runpod_runbook.md` section H). Validation is the fixed 50k split; its checkpoints are rated later with
Step 12's per-game records, so the eval cell is skipped (`RUN_EVAL=0`). The notebook uses the repo clone it is
opened from.

**Safe to interrupt.** Training resumes from the last checkpoint. Evaluation overwrites each checkpoint's
row in `results.csv` after every batch of games, so stopping early keeps the latest Elo + CI.

**Needs a GPU** (L4 pod).

In [ ]:
# Setup (RunPod): the repo lives on the volume; artifacts under artifact_root (configs/base.yaml).
# Before each session, in a pod terminal (code is never edited on the pod):
#   cd /workspace/Chess_Bot && git fetch && git checkout BRANCH && git reset --hard origin/BRANCH
import os
# The clone this notebook was opened from (its notebooks/ folder's parent), so a second clone,
# e.g. /workspace/Chess_Bot_sft on a second L4 pod, runs its own code (plan v5 Sec 4.1).
_here = os.getcwd()
REPO_DIR = os.path.dirname(_here) if os.path.basename(_here) == "notebooks" else _here
%cd {REPO_DIR}
!git log -1 --oneline
!pip install -q -r requirements.txt

import yaml
ROOT = yaml.safe_load(open("configs/base.yaml"))["artifact_root"]
import torch
from engine.stockfish import ensure_stockfish, SF_VERSION
print("Stockfish:", SF_VERSION, ensure_stockfish())
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none -- this needs a GPU pod"
CPU_MODEL = next((l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")), "unknown")
print("GPU:", GPU, "| CPU:", CPU_MODEL, "| cores:", os.cpu_count())

## Parameters
Everything comes from the config file (`SFT_CONFIG`, default `configs/sft.yaml`) except the switches below,
which are read from the shell so a headless run needs no commit (e.g. `RUN_TRAINING=1 jupyter nbconvert ...`). The run folder is named after the settings that
define the run, so rerunning with the same settings resumes it instead of starting over.

In [ ]:
import dataclasses
import json
import math

import yaml

from data.dataset import PositionDataset, load_split
from model.llm_policy import LLMPolicy, PolicyConfig
from train.sft import TrainConfig, list_checkpoints, run_name, train

_flag = lambda name, default="0": os.environ.get(name, default) == "1"
SFT_CONFIG = os.environ.get("SFT_CONFIG", "configs/sft.yaml")  # configs/sft_500k.yaml = Step 14's 500k run
RUN_TRAINING = _flag("RUN_TRAINING")  # off by default: Step 5's run is done; on only to train a new config
SMOKE_TEST = _flag("SMOKE_TEST")      # with RUN_TRAINING: a few steps on 64 positions (checks speed + memory)
RUN_EVAL = _flag("RUN_EVAL", "1")     # off for runs rated later with Step 12's per-game records

cfg = yaml.safe_load(open(SFT_CONFIG))
data_cfg = yaml.safe_load(open("configs/data.yaml"))
POLICY = PolicyConfig.from_dict(cfg["policy"])
TRAIN = TrainConfig.from_dict(cfg["train"])
FULL_TRAIN = TRAIN
if SMOKE_TEST:
    TRAIN = dataclasses.replace(TRAIN, train_positions=64, val_positions=32, val_top1_positions=8, epochs=1,
                                checkpoint_every=2, log_every=1)

RUN = run_name(POLICY, TRAIN) + ("_smoke" if SMOKE_TEST else "")
RUN_DIR = os.path.join(ROOT, "checkpoints", "sft", RUN)
DATA_DIR = os.path.join(ROOT, "data", "processed", TRAIN.dataset)
print("config:", SFT_CONFIG, "| RUN_TRAINING", RUN_TRAINING, "| SMOKE_TEST", SMOKE_TEST, "| RUN_EVAL", RUN_EVAL)
print("player:", POLICY.model_id, "|", POLICY.representation, "| masked" if POLICY.masked else "| unmasked")
print("training:", TRAIN)
print("run folder:", RUN_DIR)

## Data
The Step 3 dataset, split by game exactly as in Step 3 -- or, with `val_from` set (the 500k run), the whole
dataset for training and the 50k set's held-out games for validation. Only needed for training.

In [ ]:
if RUN_TRAINING:
    train_df, val_df = load_split(os.path.join(ROOT, "data", "processed"), TRAIN.dataset,
                                  data_cfg["val_fraction"], data_cfg["seed"], val_from=TRAIN.val_from)
    train_ds = PositionDataset(train_df, POLICY.representation)
    val_ds = PositionDataset(val_df, POLICY.representation)
    manifest = json.load(open(os.path.join(DATA_DIR, "manifest.json")))
    n_used = min(TRAIN.train_positions or len(train_ds), len(train_ds))
    sf_cpu_total = manifest["stockfish_cpu_seconds"]
    sf_cpu_used = sf_cpu_total * n_used / manifest["rows"]
    print(f"train split {len(train_ds)} | validation split {len(val_ds)} | this run trains on {n_used}")
    print(f"Stockfish labeling CPU: whole dataset {sf_cpu_total / 60:.1f} min | share for {n_used} positions {sf_cpu_used / 60:.1f} min")
else:
    print("RUN_TRAINING is False -- skipping data")

## Load the model
Qwen3-0.6B (downloaded once per session, ~1.2 GB) with fresh LoRA adapters and a zero-initialized value head.
Only those train; the base weights stay frozen.

In [ ]:
if RUN_TRAINING:
    policy = LLMPolicy.from_pretrained(POLICY)
    n_train = sum(p.numel() for p in policy.trainable_parameters())
    n_all = sum(p.numel() for p in policy.model.parameters())
    print(f"dtype {policy.model.get_input_embeddings().weight.dtype} on {policy.device}")
    print(f"trainable {n_train / 1e6:.1f}M of {n_all / 1e6:.0f}M parameters ({n_train / n_all:.1%})")
else:
    print("RUN_TRAINING is False -- skipping model load")

## Train
Prints the train losses every few steps and, at every checkpoint, the held-out numbers: move loss, value loss
(and its mean error in win probability), and **top-1** -- how often the masked pick equals Stockfish's move.
If this cell is interrupted, rerun the notebook from the top and it picks up where it stopped.

In [ ]:
if RUN_TRAINING:
    torch.cuda.reset_peak_memory_stats()
    summary = train(policy, train_ds, val_ds, TRAIN, RUN_DIR, extra_info={
        "gpu": GPU, "cpu": CPU_MODEL, "stockfish": SF_VERSION, "dataset_rows": manifest["rows"],
        "stockfish_cpu_seconds_dataset": sf_cpu_total, "stockfish_cpu_seconds_share": sf_cpu_used})
    print(json.dumps(summary, indent=2))

    per_step = summary["gpu_seconds_train"] / summary["step"]
    full_steps = FULL_TRAIN.epochs * math.ceil(n_used / FULL_TRAIN.batch_size)  # n_used: the resolved
                                                                                # count (None -> whole split)
    print(f"\n{per_step:.2f} s/step at batch {TRAIN.batch_size} | peak GPU memory {summary.get('peak_gpu_memory_gb')} GB "
          f"of {torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB")
    print(f"full run: {full_steps} steps -> ~{full_steps * per_step / 60:.0f} min of training (+ validation at each checkpoint)")
else:
    print("RUN_TRAINING is False -- skipping training")

## Training curves

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

tlog = pd.read_csv(os.path.join(RUN_DIR, "train_log.csv"))
vlog = pd.read_csv(os.path.join(RUN_DIR, "val_log.csv"))
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(tlog["step"], tlog["train_move_loss"], label="train")
ax[0].plot(vlog["step"], vlog["val_move_loss"], "o-", label="validation")
ax[0].set_title("move loss: -log P(Stockfish's move)")
ax[1].plot(tlog["step"], tlog["train_value_loss"], label="train")
ax[1].plot(vlog["step"], vlog["val_value_loss"], "o-", label="validation")
ax[1].set_title(f"value loss ({TRAIN.value_loss})")
ax[2].plot(vlog["step"], vlog["val_top1"], "o-")
ax[2].set_title("top-1: masked pick == Stockfish's move")
for a in ax:
    a.set_xlabel("step")
    a.legend() if a.get_legend_handles_labels()[0] else None
plt.tight_layout()
plt.show()
print(vlog.to_string(index=False))

## Evaluate checkpoints into `results.csv`

Pick 3-4 checkpoints spread over the run -- they're the points on this run's Elo-vs-compute curve -- and the
2-3 reference members nearest their expected strength (see the fixed ratings printed below). Each plays
`games_per_opponent` games per opponent (`configs/eval.yaml`, 20) in batches of `batch_games` (10); after every
batch its row in `results.csv` is overwritten. No CI target: report what you get, and add games by hand only
if a CI is wider than +/-200. A checkpoint that already has its full game count is skipped on a rerun.

In [ ]:
if RUN_EVAL:
    from eval.anchors import ResultsTable, build_reference_agents, close_agents
    from eval.elo import format_table
    from eval.match import evaluate_agent
    from train.sft import gpu_hours_at, list_checkpoints, load_checkpoint

    eval_cfg = yaml.safe_load(open("configs/eval.yaml"))
    base_cfg = yaml.safe_load(open("configs/base.yaml"))
    results = ResultsTable(os.path.join(ROOT, eval_cfg["results_file"]))
    FIXED = results.fixed()
    print("fixed ratings:", {n: round(v) for n, v in sorted(FIXED.items(), key=lambda kv: kv[1])})
    # Labeling CPU charged to this run: its dataset's manifest (0.99 h for the 50k set, 3.10 h for 500k).
    _manifest = os.path.join(DATA_DIR, "manifest.json")
    LABEL_CPU_HOURS = (json.load(open(_manifest))["stockfish_cpu_seconds"] / 3600 if os.path.exists(_manifest)
                       else base_cfg["label_cpu_hours"])
    step_dirs = dict(list_checkpoints(RUN_DIR))
    print("checkpoints:", sorted(step_dirs))

    # Edit before running. step_20000 is itself a reference member, so it isn't re-evaluated here.
    EVAL_STEPS = [5000, 10000, 15000, 25000]
    OPPONENTS = ["greedy", "step_20000", "sf_1320"]
    GAMES, BATCH = eval_cfg["games_per_opponent"], eval_cfg["batch_games"]

    opponents = build_reference_agents(results, OPPONENTS, ROOT)
    try:
        for step in EVAL_STEPS:
            name = f"{RUN}@{step}"
            if results.role(name) == "model" and int(results.rows[name]["games"]) >= GAMES * len(OPPONENTS):
                print(f"{name}: already evaluated ({results.rows[name]['games']} games) -- skipped")
                continue
            agent = load_checkpoint(step_dirs[step], name=name)

            def save(est, recs, step=step):
                results.log_model(est, recs, run=RUN, step=step, seed=TRAIN.seed,
                                  train_gpu_hours=gpu_hours_at(RUN_DIR, step),
                                  label_cpu_hours=LABEL_CPU_HOURS, gpu=GPU)
                results.save()
                print(format_table([est]))

            evaluate_agent(agent, opponents, FIXED, GAMES, batch_games=BATCH, on_batch=save)
            del agent
            torch.cuda.empty_cache()
    finally:
        close_agents(opponents)

    import pandas as pd
    display(pd.read_csv(results.path))
else:
    print("RUN_EVAL is off -- checkpoints are rated later (Step 12 per-game records)")


## Elo vs. compute
Every run in `results.csv` on one chart: SFT checkpoints (`step_20000` as a hollow marker), each RL run in its
own color joined to the SFT checkpoint it started from, 95% CIs, nearby reference ratings as dashed lines.
Saved to `results/elo_vs_compute.png` on the volume (`eval/plots.py`, reads only `results.csv`).

In [ ]:
from eval.plots import plot_elo_vs_compute

RESULTS_FILE = os.path.join(ROOT, yaml.safe_load(open("configs/eval.yaml"))["results_file"])
PLOT_FILE = os.path.join(ROOT, "results", "elo_vs_compute.png")
fig = plot_elo_vs_compute(RESULTS_FILE, PLOT_FILE)
display(fig)
print("saved:", PLOT_FILE)